In [10]:
import os 
import json
import pandas as pd


In [19]:
food_df = pd.read_csv('data/compliant_user_items.csv')

# Load JSON file containing food corrections
with open('/Users/vince/Salk/mCC_Analysis/food_corrections.json', 'r') as f:
    food_corrections = json.load(f)

# Create a new column with corrected food names
food_df['corrected_food'] = food_df['parsing_result'].map(food_corrections).str.lower()

food_df = food_df[['pid', 'original_logtime','log_date', 'time', 'corrected_food']]


In [ ]:
sleep_df = pd.read_csv('data/compliant_user_sleep.csv', index_col = 0)
sleep_df = sleep_df[(sleep_df['sleep_duration_decimal'] > 2) & 
                                 (sleep_df['sleep_duration_decimal'] < 14)]


sleep_df



,pid,sleep_time,wakeup_time,sleep_duration,measured_date,enough_sleep,sleep_problems,sleep_time_decimal,sleep_duration_decimal,wakeup_time_decimal
11,alqt211031188312,23:00:00,06:00:00,07:00:00,2021-11-01,True,NaN,23.000000,7.000000,6.000000
12,alqt211031188312,23:00:00,06:00:00,07:00:00,2021-11-01,False,NaN,23.000000,7.000000,6.000000
16,alqt211051189149,22:49:00,06:39:00,07:50:00,2021-11-01,False,NaN,22.816667,7.833333,6.650000
38,alqt210850183819,21:04:00,03:00:00,05:56:00,2021-11-01,True,NaN,21.066667,5.933333,3.000000
40,alqt211018190512,04:06:00,06:35:00,02:29:00,2021-11-01,True,NaN,4.100000,2.483333,6.583333
...,...,...,...,...,...,...,...,...,...,...
1142020,alqt150230644249722,01:16:00,10:28:00,09:12:00,2024-02-18,False,--5.0,1.266667,9.200000,10.466667
1142021,alqt150230954258083,21:51:00,04:30:00,06:39:00,2024-02-18,False,watched TV too late--0.0,21.850000,6.650000,4.500000
1142022,alqt16075004,21:00:00,04:00:00,07:00:00,2024-02-18,True,--3.0,21.000000,7.000000,4.000000
1142023,alqt170410595,19:54:00,02:46:00,06:52:00,2024-02-18,False,--1.0,19.900000,6.866667,2.766667


In [21]:
duplicate_cols = ['pid', 'measured_date', 'sleep_time', 'wakeup_time', 'sleep_duration']
duplicate_mask = sleep_df.duplicated(subset=duplicate_cols, keep='last')
sleep_df_deduped = sleep_df[~duplicate_mask]
sleep_df_deduped.shape


(714645, 10)

In [22]:
from datetime import datetime, timedelta

clean_data = []

for (pid, date), group in sleep_df_deduped.groupby(['pid', 'measured_date']):
    if len(group) == 1:
        # Only one entry, keep it
        clean_data.append(group)
    else:
        # Simply keep the row with the highest index
        # This assumes that the index represents the order of entry
        last_entry = group.iloc[[group.index.argmax()]]
        clean_data.append(last_entry)

# Combine the results
cleaned_sleep_df = pd.concat(clean_data)

# Show the results
print(f"Original shape: {sleep_df.shape}")
print(f"After deduplication: {sleep_df_deduped.shape}")
print(f"After resolving overlaps: {cleaned_sleep_df.shape}")

Original shape: (722794, 10)
After deduplication: (714645, 10)
After resolving overlaps: (690632, 10)


In [23]:
cleaned_sleep_df

,pid,sleep_time,wakeup_time,sleep_duration,measured_date,enough_sleep,sleep_problems,sleep_time_decimal,sleep_duration_decimal,wakeup_time_decimal
294626,alqt150211047,22:35:00,06:30:00,07:55:00,2021-10-02,False,Difficult falling asleep--separator--Woke up o...,22.583333,7.916667,6.500000
294627,alqt150211047,23:10:00,08:08:00,08:58:00,2021-10-28,False,Difficult falling asleep,23.166667,8.966667,8.133333
294628,alqt150211047,23:30:00,09:00:00,09:30:00,2021-10-29,True,NaN,23.500000,9.500000,9.000000
294629,alqt150211047,22:20:00,07:15:00,08:55:00,2021-10-30,False,Difficult falling asleep--separator--Woke up o...,22.333333,8.916667,7.250000
294630,alqt150211047,22:30:00,07:33:00,09:03:00,2021-10-31,True,NaN,22.500000,9.050000,7.550000
...,...,...,...,...,...,...,...,...,...,...
1123186,alqt230941256543,22:30:00,06:55:00,08:25:00,2023-09-26,False,--1.0,22.500000,8.416667,6.916667
1123366,alqt230941256543,02:15:00,07:30:00,05:15:00,2023-09-27,False,--1.0,2.250000,5.250000,7.500000
1123504,alqt230941256543,00:10:00,08:30:00,08:20:00,2023-09-28,False,--1.0,0.166667,8.333333,8.500000
1123568,alqt230941256543,22:30:00,05:04:00,06:34:00,2023-09-29,False,--1.0,22.500000,6.566667,5.066667


In [26]:
import pandas as pd

# Assuming your dataframe is named cleaned_sleep_df
# Make a copy to avoid modifying the original DataFrame directly if needed
df = cleaned_sleep_df.copy()

# --- 1. Ensure Correct Data Types ---
# Convert measured_date to datetime objects if it's not already
df['measured_date'] = pd.to_datetime(df['measured_date'])

# Ensure time columns are strings (they usually are when read from CSV)
df['sleep_time'] = df['sleep_time'].astype(str)
df['wakeup_time'] = df['wakeup_time'].astype(str)

# --- 2. Combine Date and Time (Initial) ---
# Combine measured_date (as string) with time strings, then convert to datetime
# Using strftime ensures a consistent date format before combining
date_str_format = '%Y-%m-%d'
try:
    df['sleep_datetime'] = pd.to_datetime(
        df['measured_date'].dt.strftime(date_str_format) + ' ' + df['sleep_time'],
        errors='coerce' # Turns parsing errors into NaT (Not a Time)
    )
    # Initially assume wakeup is on the same measured_date
    df['wakeup_datetime'] = pd.to_datetime(
        df['measured_date'].dt.strftime(date_str_format) + ' ' + df['wakeup_time'],
        errors='coerce'
    )
except Exception as e:
    print(f"Error during initial datetime conversion: {e}")
    print("Please check your time formats in 'sleep_time' and 'wakeup_time'. Expected like 'HH:MM:SS'")
    # Handle error appropriately, maybe exit or raise

# --- 3. Identify "Next Day" Wakeups ---
# Find rows where the initial wakeup_datetime is earlier than sleep_datetime
# This implies the day rolled over. Also handle NaT values.
next_day_condition = (df['wakeup_datetime'].notna()) & \
                     (df['sleep_datetime'].notna()) & \
                     (df['wakeup_datetime'] < df['sleep_datetime'])

# --- 4. Adjust Wakeup Datetime ---
# Add one day to wakeup_datetime for the rows identified above
df.loc[next_day_condition, 'wakeup_datetime'] = df.loc[next_day_condition, 'wakeup_datetime'] + pd.Timedelta(days=1)

# --- Optional: Verify and Display ---
print("DataFrame with adjusted wakeup_datetime:")
display(df[['pid', 'measured_date', 'sleep_time', 'wakeup_time', 'sleep_datetime', 'wakeup_datetime']].head())

# --- Optional: Calculate actual duration (more accurate) ---
# This can serve as a sanity check against your existing 'sleep_duration'
df['calculated_duration_seconds'] = (df['wakeup_datetime'] - df['sleep_datetime']).dt.total_seconds()
df['calculated_duration_hours'] = df['calculated_duration_seconds'] / 3600

print("\nVerification with calculated duration:")
display(df[['sleep_duration', 'sleep_duration_decimal', 'calculated_duration_hours']].head())

# You can now use 'sleep_datetime' and 'wakeup_datetime' for further analysis.
# You might want to drop the intermediate or original time columns if no longer needed.
# cleaned_sleep_df = df # Assign back if you want to overwrite the original variable

DataFrame with adjusted wakeup_datetime:


,pid,measured_date,sleep_time,wakeup_time,sleep_datetime,wakeup_datetime
294626,alqt150211047,2021-10-02,22:35:00,06:30:00,2021-10-02 22:35:00,2021-10-03 06:30:00
294627,alqt150211047,2021-10-28,23:10:00,08:08:00,2021-10-28 23:10:00,2021-10-29 08:08:00
294628,alqt150211047,2021-10-29,23:30:00,09:00:00,2021-10-29 23:30:00,2021-10-30 09:00:00
294629,alqt150211047,2021-10-30,22:20:00,07:15:00,2021-10-30 22:20:00,2021-10-31 07:15:00
294630,alqt150211047,2021-10-31,22:30:00,07:33:00,2021-10-31 22:30:00,2021-11-01 07:33:00



Verification with calculated duration:


,sleep_duration,sleep_duration_decimal,calculated_duration_hours
294626,07:55:00,7.916667,7.916667
294627,08:58:00,8.966667,8.966667
294628,09:30:00,9.500000,9.500000
294629,08:55:00,8.916667,8.916667
294630,09:03:00,9.050000,9.050000


In [27]:
df

,pid,sleep_time,wakeup_time,sleep_duration,measured_date,enough_sleep,sleep_problems,sleep_time_decimal,sleep_duration_decimal,wakeup_time_decimal,sleep_datetime,wakeup_datetime,calculated_duration_seconds,calculated_duration_hours
294626,alqt150211047,22:35:00,06:30:00,07:55:00,2021-10-02,False,Difficult falling asleep--separator--Woke up o...,22.583333,7.916667,6.500000,2021-10-02 22:35:00,2021-10-03 06:30:00,28500.0,7.916667
294627,alqt150211047,23:10:00,08:08:00,08:58:00,2021-10-28,False,Difficult falling asleep,23.166667,8.966667,8.133333,2021-10-28 23:10:00,2021-10-29 08:08:00,32280.0,8.966667
294628,alqt150211047,23:30:00,09:00:00,09:30:00,2021-10-29,True,NaN,23.500000,9.500000,9.000000,2021-10-29 23:30:00,2021-10-30 09:00:00,34200.0,9.500000
294629,alqt150211047,22:20:00,07:15:00,08:55:00,2021-10-30,False,Difficult falling asleep--separator--Woke up o...,22.333333,8.916667,7.250000,2021-10-30 22:20:00,2021-10-31 07:15:00,32100.0,8.916667
294630,alqt150211047,22:30:00,07:33:00,09:03:00,2021-10-31,True,NaN,22.500000,9.050000,7.550000,2021-10-31 22:30:00,2021-11-01 07:33:00,32580.0,9.050000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1123186,alqt230941256543,22:30:00,06:55:00,08:25:00,2023-09-26,False,--1.0,22.500000,8.416667,6.916667,2023-09-26 22:30:00,2023-09-27 06:55:00,30300.0,8.416667
1123366,alqt230941256543,02:15:00,07:30:00,05:15:00,2023-09-27,False,--1.0,2.250000,5.250000,7.500000,2023-09-27 02:15:00,2023-09-27 07:30:00,18900.0,5.250000
1123504,alqt230941256543,00:10:00,08:30:00,08:20:00,2023-09-28,False,--1.0,0.166667,8.333333,8.500000,2023-09-28 00:10:00,2023-09-28 08:30:00,30000.0,8.333333
1123568,alqt230941256543,22:30:00,05:04:00,06:34:00,2023-09-29,False,--1.0,22.500000,6.566667,5.066667,2023-09-29 22:30:00,2023-09-30 05:04:00,23640.0,6.566667


In [34]:
food_pid = set(food_df['pid'].unique())
sleep_pid = set(df['pid'].unique())
intersection_pid = food_pid.intersection(sleep_pid)

In [ ]:
food_pid['pid']

In [42]:
import pandas as pd

# Assume 'df' is your cleaned sleep dataframe from the previous step
# with 'sleep_datetime' and 'wakeup_datetime' columns
# Assume 'food_df' is your food dataframe

# --- 1. Prepare DataFrames ---

# Ensure correct datetime types
df['sleep_datetime'] = pd.to_datetime(df['sleep_datetime'])
df['wakeup_datetime'] = pd.to_datetime(df['wakeup_datetime'])
food_df['food_datetime'] = pd.to_datetime(food_df['original_logtime'], errors='coerce')

# Drop rows with invalid dates/pids critical for merging
df.dropna(subset=['pid', 'sleep_datetime', 'wakeup_datetime'], inplace=True)
food_df.dropna(subset=['pid', 'food_datetime'], inplace=True)

# Find intersecting pids
food_pid = set(food_df['pid'].unique())
sleep_pid = set(df['pid'].unique())
intersection_pid = food_pid.intersection(sleep_pid)

# Filter both dataframes
df_filtered = df[df['pid'].isin(intersection_pid)].copy()
food_filtered = food_df[food_df['pid'].isin(intersection_pid)].copy()

print(f"Found {len(intersection_pid)} users in both datasets.")
print(f"Sleep records after filtering: {len(df_filtered)}")
print(f"Food records after filtering: {len(food_filtered)}")


Found 20195 users in both datasets.
Sleep records after filtering: 690632
Food records after filtering: 3077552


In [43]:
df_filtered

,pid,sleep_time,wakeup_time,sleep_duration,measured_date,enough_sleep,sleep_problems,sleep_time_decimal,sleep_duration_decimal,wakeup_time_decimal,sleep_datetime,wakeup_datetime,calculated_duration_seconds,calculated_duration_hours
294626,alqt150211047,22:35:00,06:30:00,07:55:00,2021-10-02,False,Difficult falling asleep--separator--Woke up o...,22.583333,7.916667,6.500000,2021-10-02 22:35:00,2021-10-03 06:30:00,28500.0,7.916667
294627,alqt150211047,23:10:00,08:08:00,08:58:00,2021-10-28,False,Difficult falling asleep,23.166667,8.966667,8.133333,2021-10-28 23:10:00,2021-10-29 08:08:00,32280.0,8.966667
294628,alqt150211047,23:30:00,09:00:00,09:30:00,2021-10-29,True,NaN,23.500000,9.500000,9.000000,2021-10-29 23:30:00,2021-10-30 09:00:00,34200.0,9.500000
294629,alqt150211047,22:20:00,07:15:00,08:55:00,2021-10-30,False,Difficult falling asleep--separator--Woke up o...,22.333333,8.916667,7.250000,2021-10-30 22:20:00,2021-10-31 07:15:00,32100.0,8.916667
294630,alqt150211047,22:30:00,07:33:00,09:03:00,2021-10-31,True,NaN,22.500000,9.050000,7.550000,2021-10-31 22:30:00,2021-11-01 07:33:00,32580.0,9.050000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1123186,alqt230941256543,22:30:00,06:55:00,08:25:00,2023-09-26,False,--1.0,22.500000,8.416667,6.916667,2023-09-26 22:30:00,2023-09-27 06:55:00,30300.0,8.416667
1123366,alqt230941256543,02:15:00,07:30:00,05:15:00,2023-09-27,False,--1.0,2.250000,5.250000,7.500000,2023-09-27 02:15:00,2023-09-27 07:30:00,18900.0,5.250000
1123504,alqt230941256543,00:10:00,08:30:00,08:20:00,2023-09-28,False,--1.0,0.166667,8.333333,8.500000,2023-09-28 00:10:00,2023-09-28 08:30:00,30000.0,8.333333
1123568,alqt230941256543,22:30:00,05:04:00,06:34:00,2023-09-29,False,--1.0,22.500000,6.566667,5.066667,2023-09-29 22:30:00,2023-09-30 05:04:00,23640.0,6.566667


In [44]:

# --- Robust Preparation ---
print("\n--- Robust Preparation ---")

# 1. Check for NaNs in 'pid'
print(f"NaNs in df_filtered['pid']: {df_filtered['pid'].isna().sum()}")
print(f"NaNs in food_filtered['pid']: {food_filtered['pid'].isna().sum()}")
# If > 0, consider dropping them:
# df_filtered.dropna(subset=['pid'], inplace=True)
# food_filtered.dropna(subset=['pid'], inplace=True)

# 2. Explicit Type Casting for 'pid'
print("Casting 'pid' to string type...")
try:
    df_filtered['pid'] = df_filtered['pid'].astype(str)
    food_filtered['pid'] = food_filtered['pid'].astype(str)
except Exception as e:
    print(f"Error casting pid to string: {e}")
    # Handle error if casting fails

# Ensure datetime columns are consistently typed AFTER potential drops/casts
df_filtered['wakeup_datetime'] = pd.to_datetime(df_filtered['wakeup_datetime'])
food_filtered['food_datetime'] = pd.to_datetime(food_filtered['food_datetime'])



--- Robust Preparation ---
NaNs in df_filtered['pid']: 0
NaNs in food_filtered['pid']: 0
Casting 'pid' to string type...


In [45]:

# 3. Sort and Reset Index
print("Sorting dataframes...")
df_filtered = df_filtered.sort_values(by=['pid', 'wakeup_datetime']).reset_index(drop=True)
food_filtered = food_filtered.sort_values(by=['pid', 'food_datetime']).reset_index(drop=True)


Sorting dataframes...


In [46]:
df_filtered

,pid,sleep_time,wakeup_time,sleep_duration,measured_date,enough_sleep,sleep_problems,sleep_time_decimal,sleep_duration_decimal,wakeup_time_decimal,sleep_datetime,wakeup_datetime,calculated_duration_seconds,calculated_duration_hours
0,alqt150211047,22:35:00,06:30:00,07:55:00,2021-10-02,False,Difficult falling asleep--separator--Woke up o...,22.583333,7.916667,6.500000,2021-10-02 22:35:00,2021-10-03 06:30:00,28500.0,7.916667
1,alqt150211047,23:10:00,08:08:00,08:58:00,2021-10-28,False,Difficult falling asleep,23.166667,8.966667,8.133333,2021-10-28 23:10:00,2021-10-29 08:08:00,32280.0,8.966667
2,alqt150211047,23:30:00,09:00:00,09:30:00,2021-10-29,True,NaN,23.500000,9.500000,9.000000,2021-10-29 23:30:00,2021-10-30 09:00:00,34200.0,9.500000
3,alqt150211047,22:20:00,07:15:00,08:55:00,2021-10-30,False,Difficult falling asleep--separator--Woke up o...,22.333333,8.916667,7.250000,2021-10-30 22:20:00,2021-10-31 07:15:00,32100.0,8.916667
4,alqt150211047,22:30:00,07:33:00,09:03:00,2021-10-31,True,NaN,22.500000,9.050000,7.550000,2021-10-31 22:30:00,2021-11-01 07:33:00,32580.0,9.050000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
690627,alqt230941256543,22:30:00,06:55:00,08:25:00,2023-09-26,False,--1.0,22.500000,8.416667,6.916667,2023-09-26 22:30:00,2023-09-27 06:55:00,30300.0,8.416667
690628,alqt230941256543,02:15:00,07:30:00,05:15:00,2023-09-27,False,--1.0,2.250000,5.250000,7.500000,2023-09-27 02:15:00,2023-09-27 07:30:00,18900.0,5.250000
690629,alqt230941256543,00:10:00,08:30:00,08:20:00,2023-09-28,False,--1.0,0.166667,8.333333,8.500000,2023-09-28 00:10:00,2023-09-28 08:30:00,30000.0,8.333333
690630,alqt230941256543,22:30:00,05:04:00,06:34:00,2023-09-29,False,--1.0,22.500000,6.566667,5.066667,2023-09-29 22:30:00,2023-09-30 05:04:00,23640.0,6.566667


In [47]:

# 4. Granular Sorting Check Function
def check_sorting_detailed(df, group_col, sort_col, df_name="DataFrame"):
    all_sorted = True
    offending_groups = []
    if df.empty:
        print(f"{df_name} is empty, skipping detailed sort check.")
        return True
    print(f"Performing detailed sort check on {df_name} by {group_col}, then {sort_col}...")
    for name, group in df.groupby(group_col, sort=False): # Use sort=False for efficiency
            # Use is_monotonic_increasing which is efficient
        if not group[sort_col].is_monotonic_increasing:
            all_sorted = False
            offending_groups.append(name)
            # Optional: Break early if you only need to know if *any* group failed
            # break
    if not all_sorted:
        print(f"ERROR: Detailed check failed for {df_name}. Offending groups (first few): {offending_groups[:5]}")
    else:
            print(f"Detailed check passed for {df_name}.")
    return all_sorted

# Run the detailed check
food_sorted_ok = check_sorting_detailed(food_filtered, 'pid', 'food_datetime', 'food_filtered')
sleep_sorted_ok = check_sorting_detailed(df_filtered, 'pid', 'wakeup_datetime', 'df_filtered')

# Stop if detailed check failed
if not food_sorted_ok or not sleep_sorted_ok:
    raise ValueError("Detailed sorting check failed. Cannot proceed with merge_asof.")


Performing detailed sort check on food_filtered by pid, then food_datetime...
Detailed check passed for food_filtered.
Performing detailed sort check on df_filtered by pid, then wakeup_datetime...
Detailed check passed for df_filtered.


In [49]:

# Ensure data is sorted AGAIN right before merge as a paranoid check
food_filtered_final = food_filtered.sort_values(by=['pid', 'food_datetime'])
df_filtered_final = df_filtered.sort_values(by=['pid', 'wakeup_datetime'])




In [52]:
food_filtered_final

,pid,original_logtime,log_date,time,corrected_food,food_datetime
0,alqt150211047,2021-10-28 09:45:59,2021-10-28,9.766389,nespresso,2021-10-28 09:45:59
1,alqt150211047,2021-10-28 09:45:59,2021-10-28,9.766389,oatmeal,2021-10-28 09:45:59
2,alqt150211047,2021-10-28 09:45:59,2021-10-28,9.766389,milk,2021-10-28 09:45:59
3,alqt150211047,2021-10-28 11:57:00,2021-10-28,11.950000,eggplant,2021-10-28 11:57:00
4,alqt150211047,2021-10-28 11:57:00,2021-10-28,11.950000,lasagna,2021-10-28 11:57:00
...,...,...,...,...,...,...
3077547,alqt230941256543,2023-09-21 16:32:00,2023-09-21,16.533333,hibiscus tea,2023-09-21 16:32:00
3077548,alqt230941256543,2023-09-21 16:32:00,2023-09-21,16.533333,blueberry,2023-09-21 16:32:00
3077549,alqt230941256543,2023-09-21 19:25:00,2023-09-21,19.416667,baked chicken,2023-09-21 19:25:00
3077550,alqt230941256543,2023-09-21 19:25:00,2023-09-21,19.416667,mashed potato,2023-09-21 19:25:00


In [55]:
df_filtered_final

,pid,sleep_time,wakeup_time,sleep_duration,measured_date,enough_sleep,sleep_problems,sleep_time_decimal,sleep_duration_decimal,wakeup_time_decimal,sleep_datetime,wakeup_datetime,calculated_duration_seconds,calculated_duration_hours
0,alqt150211047,22:35:00,06:30:00,07:55:00,2021-10-02,False,Difficult falling asleep--separator--Woke up o...,22.583333,7.916667,6.500000,2021-10-02 22:35:00,2021-10-03 06:30:00,28500.0,7.916667
1,alqt150211047,23:10:00,08:08:00,08:58:00,2021-10-28,False,Difficult falling asleep,23.166667,8.966667,8.133333,2021-10-28 23:10:00,2021-10-29 08:08:00,32280.0,8.966667
2,alqt150211047,23:30:00,09:00:00,09:30:00,2021-10-29,True,NaN,23.500000,9.500000,9.000000,2021-10-29 23:30:00,2021-10-30 09:00:00,34200.0,9.500000
3,alqt150211047,22:20:00,07:15:00,08:55:00,2021-10-30,False,Difficult falling asleep--separator--Woke up o...,22.333333,8.916667,7.250000,2021-10-30 22:20:00,2021-10-31 07:15:00,32100.0,8.916667
4,alqt150211047,22:30:00,07:33:00,09:03:00,2021-10-31,True,NaN,22.500000,9.050000,7.550000,2021-10-31 22:30:00,2021-11-01 07:33:00,32580.0,9.050000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
690627,alqt230941256543,22:30:00,06:55:00,08:25:00,2023-09-26,False,--1.0,22.500000,8.416667,6.916667,2023-09-26 22:30:00,2023-09-27 06:55:00,30300.0,8.416667
690628,alqt230941256543,02:15:00,07:30:00,05:15:00,2023-09-27,False,--1.0,2.250000,5.250000,7.500000,2023-09-27 02:15:00,2023-09-27 07:30:00,18900.0,5.250000
690629,alqt230941256543,00:10:00,08:30:00,08:20:00,2023-09-28,False,--1.0,0.166667,8.333333,8.500000,2023-09-28 00:10:00,2023-09-28 08:30:00,30000.0,8.333333
690630,alqt230941256543,22:30:00,05:04:00,06:34:00,2023-09-29,False,--1.0,22.500000,6.566667,5.066667,2023-09-29 22:30:00,2023-09-30 05:04:00,23640.0,6.566667


In [57]:
import pandas as pd

# --- Assuming df_filtered_final and food_filtered_final are prepared ---
# df_filtered_final MUST be sorted by pid, then wakeup_datetime
# food_filtered_final MUST be sorted by pid, then food_datetime

# Ensure sorting just in case
df_filtered_final = df_filtered_final.sort_values(by=['pid', 'wakeup_datetime']).reset_index(drop=True)
food_filtered_final = food_filtered_final.sort_values(by=['pid', 'food_datetime']).reset_index(drop=True)

# --- Define Waking Intervals Correctly ---
print("Defining waking intervals (Wakeup to Next Sleep)...")
try:
    # Ensure no NaNs in relevant columns
    sleep_boundaries = df_filtered_final[['pid', 'wakeup_datetime', 'sleep_datetime']].dropna().copy()

    # Find the start time of the NEXT sleep period for each user
    # Shift the sleep_datetime column up by 1 within each pid group
    sleep_boundaries['next_sleep_datetime'] = sleep_boundaries.groupby('pid')['sleep_datetime'].shift(-1)

    # Now create the interval: Starts at wakeup, ends at the start of the next sleep
    # Filter out rows where next_sleep_datetime is NaN (last record for each user)
    # Also filter out rows where wakeup_datetime >= next_sleep_datetime (shouldn't happen with shift if data is clean, but good check)
    valid_waking_boundaries = sleep_boundaries.dropna(subset=['next_sleep_datetime'])
    valid_waking_boundaries = valid_waking_boundaries[valid_waking_boundaries['wakeup_datetime'] < valid_waking_boundaries['next_sleep_datetime']].copy()

    # Create the Interval objects for the WAKING period
    valid_waking_boundaries['waking_interval'] = [
        pd.Interval(left, right, closed='left') for left, right in zip(valid_waking_boundaries['wakeup_datetime'], valid_waking_boundaries['next_sleep_datetime'])
    ]

    # Create the IntervalIndex map per user (for efficient lookup)
    waking_intervals_map = {}
    # Group by pid on the dataframe that now has the correct waking_interval
    for pid, group in valid_waking_boundaries.groupby('pid'):
         # Check for overlapping WAKING intervals (less likely but possible with bad data)
        # Need to sort intervals by left boundary first for IntervalIndex
        sorted_intervals = group['waking_interval'].sort_values()
        if not pd.IntervalIndex(sorted_intervals).is_overlapping:
             waking_intervals_map[pid] = pd.IntervalIndex(sorted_intervals, name='waking_interval')
        else:
             print(f"Warning: Overlapping WAKING intervals detected for pid {pid}. Check data. Skipping pid.")
             # Handle overlaps if necessary

    print(f"Created Waking IntervalIndexes for {len(waking_intervals_map)} pids.")

except Exception as e:
    print(f"Error defining waking intervals: {e}")
    raise e


# --- Map Food Data to Waking Intervals ---
print("Mapping food data to waking intervals...")

# Ensure food data is ready
food_filtered_final.dropna(subset=['pid', 'food_datetime'], inplace=True)
food_filtered_final['pid'] = food_filtered_final['pid'].astype(str) # Ensure consistent pid type
food_filtered_final['food_datetime'] = pd.to_datetime(food_filtered_final['food_datetime'])

# Function to find the interval (reusing the previous function structure)
def find_interval(row, interval_map):
    pid = row['pid']
    food_time = row['food_datetime']
    if pid in interval_map:
        interval_index = interval_map[pid]
        try:
            containing_indices = interval_index.contains(food_time)
            if containing_indices.any():
                # Using get_loc ensures we get the index, then we retrieve the interval
                # interval_loc = interval_index.get_loc(food_time) # This might error if time is exactly on boundary depending on closed side
                # Using boolean indexing is safer
                interval = interval_index[containing_indices][0] # Get the first matching interval
                return interval
        except Exception as e:
             # print(f"Error finding interval for pid {pid}, time {food_time}: {e}")
             pass
    return pd.NA

# Apply the function
food_filtered_final['waking_interval'] = food_filtered_final.apply(
    lambda row: find_interval(row, waking_intervals_map),
    axis=1
)

# --- Filter and Extract Interval Bounds ---
print("Filtering food entries within intervals...")
valid_food_intervals = food_filtered_final.dropna(subset=['waking_interval']).copy()

if valid_food_intervals.empty:
    print("No food entries found within any valid waking interval.")
else:
    # Extract interval bounds (Wakeup and Next Sleep)
    valid_food_intervals['interval_start_wake'] = valid_food_intervals['waking_interval'].apply(lambda x: x.left)
    valid_food_intervals['interval_end_sleep'] = valid_food_intervals['waking_interval'].apply(lambda x: x.right)

    print(f"Found {len(valid_food_intervals)} food entries within valid waking intervals.")

    # --- 4. Group and Aggregate ---
    print("Aggregating food times per interval...")
    # Group by the actual interval boundaries
    waking_period_stats = valid_food_intervals.groupby(['pid', 'interval_start_wake', 'interval_end_sleep']).agg(
        first_food_time=('food_datetime', 'min'),
        last_food_time=('food_datetime', 'max')
    ).reset_index()
    # Rename columns for clarity in calculations
    waking_period_stats.rename(columns={'interval_start_wake': 'wakeup_datetime',
                                        'interval_end_sleep': 'sleep_datetime'}, inplace=True)


    # --- 5. Calculate Timings per Period ---
    print("Calculating timings...")
    # Now these calculations use the correct interval boundaries
    waking_period_stats['time_wake_to_first_food'] = (
        waking_period_stats['first_food_time'] - waking_period_stats['wakeup_datetime']
    )
    waking_period_stats['time_wake_to_last_food'] = (
        waking_period_stats['last_food_time'] - waking_period_stats['wakeup_datetime']
    )
    waking_period_stats['time_last_food_to_sleep'] = (
        waking_period_stats['sleep_datetime'] - waking_period_stats['last_food_time']
    )

    # Convert Timedeltas to hours
    waking_period_stats['hours_wake_to_first_food'] = \
        waking_period_stats['time_wake_to_first_food'].dt.total_seconds() / 3600
    waking_period_stats['hours_wake_to_last_food'] = \
        waking_period_stats['time_wake_to_last_food'].dt.total_seconds() / 3600
    waking_period_stats['hours_last_food_to_sleep'] = \
        waking_period_stats['time_last_food_to_sleep'].dt.total_seconds() / 3600

    # Clip negative durations (e.g., food exactly at sleep time)
    waking_period_stats['hours_last_food_to_sleep'] = waking_period_stats['hours_last_food_to_sleep'].clip(lower=0)


    print("\nSample Waking Period Stats:")
    print(waking_period_stats.head())

    # --- 6. Calculate Mean Timings ---
    print("Calculating mean timings...")
    mean_timings = {
        'mean_hours_wake_to_first_food': waking_period_stats['hours_wake_to_first_food'].mean(skipna=True),
        'mean_hours_wake_to_last_food': waking_period_stats['hours_wake_to_last_food'].mean(skipna=True),
        'mean_hours_last_food_to_sleep': waking_period_stats['hours_last_food_to_sleep'].mean(skipna=True)
    }

    print("\nOverall Mean Timings (in hours):")
    for key, value in mean_timings.items():
        print(f"{key}: {value:.2f}")

    # Optional: Calculate mean timings per user
    mean_timings_per_user = waking_period_stats.groupby('pid')[[
        'hours_wake_to_first_food',
        'hours_wake_to_last_food',
        'hours_last_food_to_sleep'
    ]].mean(skipna=True).reset_index()

    print("\nSample Mean Timings Per User (in hours):")
    print(mean_timings_per_user.head())

Defining waking intervals (Wakeup to Next Sleep)...
Created Waking IntervalIndexes for 19923 pids.
Mapping food data to waking intervals...
Filtering food entries within intervals...
Found 2798437 food entries within valid waking intervals.
Aggregating food times per interval...
Calculating timings...

Sample Waking Period Stats:
             pid     wakeup_datetime      sleep_datetime     first_food_time  \
0  alqt150211047 2021-10-03 06:30:00 2021-10-28 23:10:00 2021-10-28 09:45:59   
1  alqt150211047 2021-10-29 08:08:00 2021-10-29 23:30:00 2021-10-29 10:30:59   
2  alqt150211047 2021-10-30 09:00:00 2021-10-30 22:20:00 2021-10-30 10:28:00   
3  alqt150211047 2021-10-31 07:15:00 2021-10-31 22:30:00 2021-10-31 09:27:00   
4  alqt150211047 2021-11-01 07:33:00 2021-11-01 23:27:00 2021-11-01 10:06:00   

       last_food_time time_wake_to_first_food time_wake_to_last_food  \
0 2021-10-28 17:05:00        25 days 03:15:59       25 days 10:35:00   
1 2021-10-29 20:07:00         0 days 02:22:

TypeError: GroupBy.mean() got an unexpected keyword argument 'skipna'

In [60]:
print("\nSample Waking Period Stats:")
display(waking_period_stats.head())


Sample Waking Period Stats:


,pid,wakeup_datetime,sleep_datetime,first_food_time,last_food_time,time_wake_to_first_food,time_wake_to_last_food,time_last_food_to_sleep,hours_wake_to_first_food,hours_wake_to_last_food,hours_last_food_to_sleep
0,alqt150211047,2021-10-03 06:30:00,2021-10-28 23:10:00,2021-10-28 09:45:59,2021-10-28 17:05:00,25 days 03:15:59,25 days 10:35:00,0 days 06:05:00,603.266389,610.583333,6.083333
1,alqt150211047,2021-10-29 08:08:00,2021-10-29 23:30:00,2021-10-29 10:30:59,2021-10-29 20:07:00,0 days 02:22:59,0 days 11:59:00,0 days 03:23:00,2.383056,11.983333,3.383333
2,alqt150211047,2021-10-30 09:00:00,2021-10-30 22:20:00,2021-10-30 10:28:00,2021-10-30 19:02:00,0 days 01:28:00,0 days 10:02:00,0 days 03:18:00,1.466667,10.033333,3.300000
3,alqt150211047,2021-10-31 07:15:00,2021-10-31 22:30:00,2021-10-31 09:27:00,2021-10-31 17:40:00,0 days 02:12:00,0 days 10:25:00,0 days 04:50:00,2.200000,10.416667,4.833333
4,alqt150211047,2021-11-01 07:33:00,2021-11-01 23:27:00,2021-11-01 10:06:00,2021-11-01 17:16:00,0 days 02:33:00,0 days 09:43:00,0 days 06:11:00,2.550000,9.716667,6.183333


In [63]:
waking_period_stats[waking_period_stats['hours_wake_to_first_food'] < 48].to_csv('food_sleep_intervals.csv')